# Phase 2 — Signal Detection (single-population, EB shrinkage)

This phase converts raw FAERS reports from Phase 1 into per-drug-event safety signals.
It computes the standard pharmacovigilance disproportionality metrics (PRR, ROR,
chi-square) and the FDA-style Empirical Bayes shrinkage (EBGM, EB05), then flags pairs
that exceed the signal threshold.

### v5 → v6 — full re-engineering of the statistics engine

1. **Single-population 2x2 tables.** `a`, the event marginals, and `N` all come from the
   same Phase 1 database. This removes the population mismatch in v5 that caused PRR
   values of 10⁸ and ROR values of 10¹⁶.
2. **Haldane-Anscombe continuity correction (+0.5).** Eliminates infinite/collapsed
   ratios in zero-cell tables.
3. **True empirical-Bayes EBGM / EB05** via Gamma-Poisson shrinkage (DuMouchel's
   single-component MGPS approximation). Small-count signals (a=3) are shrunk toward
   the prior, suppressing noise. (Replaces v5's pseudo-EBGM = a/E.)
4. **Sensitivity and specificity validation.** Detection rate against modern positive
   controls (ciprofloxacin / isotretinoin / clozapine / warfarin) plus signal rate
   on negative controls (which should be low).
5. **Optional legacy validation.** Rofecoxib / rosiglitazone validated **inside** the
   `*_legacy` population only.

> **Input:** Phase 1's `faers.db`. **Output:** `signal_results` table in the same DB,
> plus CSV exports and validation figures.

## 1. Setup & Load (single population)

Connect to the Phase 1 `faers.db`, import statistical libraries (scipy, statsmodels),
and load the total report count and per-event background marginals into memory for fast
2x2 table construction.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, sqlite3
import pandas as pd, numpy as np
from scipy import stats
from scipy.optimize import minimize
from scipy.special import gammaln, digamma
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

PROJECT_DIR="/content/drive/MyDrive/FAERS_Intelligence"
DB_PATH=os.path.join(PROJECT_DIR,"data","db","faers.db")
RESULTS_DIR=os.path.join(PROJECT_DIR,"results"); os.makedirs(RESULTS_DIR,exist_ok=True)
FIG_DIR=os.path.join(PROJECT_DIR,"figures"); os.makedirs(FIG_DIR,exist_ok=True)

conn=sqlite3.connect(DB_PATH)
meta=dict(zip(*[pd.read_sql("SELECT key,value FROM meta",conn)[c] for c in ["key","value"]]))
N_TOTAL=int(meta["n_reports_total"])
bg=pd.read_sql("SELECT pt,n_reports FROM bg_event_marginals",conn)
BG=dict(zip(bg.pt,bg.n_reports))
print(f"schema={meta.get('schema_version')} | N_total={N_TOTAL:,} | events={len(BG):,}")
print(f"quarters={meta.get('quarters')}")
conn.close()

Mounted at /content/drive
schema=phase1_v6 | N_total=1,409,287 | events=16,282
quarters=["2024q3", "2024q2", "2024q1", "2023q4"]


## 2. Configuration + Known-Signal Reference (for validation)

- `DRUG_ROLES`: which drug-role codes to include in the 2x2 tables (primary suspect PS
  and secondary suspect SS are the pharmacovigilance standard).
- `MIN_COUNT`: minimum count `a` required to test a pair (default 3).
- `EB05_SIGNAL`: signal threshold on EB05 (MGPS standard is >2).
- `KNOWN_SIGNALS`: established positive-control associations used to measure detection
  sensitivity.

In [ ]:
DRUG_ROLES=("PS","SS")      # primary / secondary suspect
MIN_COUNT=3                 # minimum a to test a pair
EB05_SIGNAL=4.0             # MGPS-standard threshold on EB05 (>2 typical, 4 stricter)

KNOWN_SIGNALS={
    # modern positive controls
    "ciprofloxacin":["TENDON RUPTURE","TENDON DISORDER","TENDONITIS","ARTHRALGIA"],
    "isotretinoin":["DEPRESSION","CHEILITIS","DRY SKIN"],
    "clozapine":["AGRANULOCYTOSIS","NEUTROPENIA","MYOCARDITIS"],
    "warfarin":["HAEMORRHAGE","INTERNATIONAL NORMALISED RATIO INCREASED","GASTROINTESTINAL HAEMORRHAGE"],
    # legacy positive controls (validated against *_legacy tables only)
    "rofecoxib":["MYOCARDIAL INFARCTION","CEREBROVASCULAR ACCIDENT"],
    "rosiglitazone":["CARDIAC FAILURE CONGESTIVE","MYOCARDIAL INFARCTION","OEDEMA PERIPHERAL"],
}
print("Config set. roles=",DRUG_ROLES,"min_count=",MIN_COUNT,"EB05>",EB05_SIGNAL)

## 3. Build 2x2 Contingency Tables (one population)

For every (drug, event) candidate we compute the four cells of the 2x2 table:

| | Drug | Not drug |
|---|---|---|
| Event Y | a | c |
| Not event Y | b | d |

- `a`: reports of drug X with event Y
- `b`: reports of drug X without event Y
- `c`: reports of event Y without drug X
- `d`: reports of neither

These counts drive every downstream disproportionality metric.

In [3]:
conn=sqlite3.connect(DB_PATH)
tgt=pd.read_sql("SELECT primaryid,target_drug,target_group,target_role,role_cod "
                "FROM drug WHERE is_target=1",conn)
reac=pd.read_sql("SELECT primaryid,pt FROM reac",conn)
conn.close()

tgt=tgt[tgt.role_cod.isin(DRUG_ROLES)].drop_duplicates(["primaryid","target_drug"])
ndrug=tgt.groupby("target_drug")["primaryid"].nunique()

de=(reac.merge(tgt[["primaryid","target_drug","target_group","target_role"]],on="primaryid")
        .drop_duplicates(["primaryid","target_drug","pt"]))
tab=(de.groupby(["target_drug","target_group","target_role","pt"])["primaryid"]
        .nunique().reset_index(name="a"))
tab["n_drug"]=tab["target_drug"].map(ndrug)
tab["n_event"]=tab["pt"].map(BG).fillna(tab["a"]).clip(lower=tab["a"])
tab["N"]=N_TOTAL
tab["b"]=tab.n_drug-tab.a
tab["c"]=tab.n_event-tab.a
tab["d"]=tab.N-tab.a-tab.b-tab.c
tab["expected"]=(tab.n_drug*tab.n_event)/tab.N
tab=tab[(tab.b>=0)&(tab.c>=0)&(tab.d>=0)].reset_index(drop=True)
print(f"Built {len(tab):,} drug-event pairs across {tab.target_drug.nunique()} drugs "
      f"(a>=1). Will report a>={MIN_COUNT}.")
print(tab.groupby('target_role')['a'].count())

Built 30,564 drug-event pairs across 18 drugs (a>=1). Will report a>=3.
target_role
neg      11000
pos       6184
study    13380
Name: a, dtype: int64


## 4. Disproportionality Metrics (PRR / ROR / chi-square) with continuity correction

Classical pharmacovigilance signal-detection statistics.

- **PRR** (Proportional Reporting Ratio) and **ROR** (Reporting Odds Ratio) with 95%
  confidence intervals.
- **Yates-corrected chi-square** with degrees of freedom 1.
- **Haldane-Anscombe correction (+0.5)** applied to all cells — eliminates the
  divide-by-zero blow-ups (PRR=10⁸, ROR=10¹⁶) that appeared in v5.

In [4]:
def add_da_metrics(df):
    a,b,c,d=df.a.values.astype(float),df.b.values.astype(float),df.c.values.astype(float),df.d.values.astype(float)
    N=a+b+c+d
    # Haldane-Anscombe +0.5 (for ratio estimators)
    a2,b2,c2,d2=a+.5,b+.5,c+.5,d+.5
    prr=(a2/(a2+b2))/(c2/(c2+d2))
    ror=(a2*d2)/(b2*c2)
    se=np.sqrt(1/a2+1/b2+1/c2+1/d2)
    df["prr"]=np.round(prr,4)
    df["ror"]=np.round(ror,4)
    df["ror_ci_lower"]=np.round(np.exp(np.log(ror)-1.96*se),4)
    df["ror_ci_upper"]=np.round(np.exp(np.log(ror)+1.96*se),4)
    # Yates-corrected chi-square on raw counts
    denom=(a+b)*(c+d)*(a+c)*(b+d)
    chi2=np.where(denom>0, N*np.clip(np.abs(a*d-b*c)-N/2,0,None)**2/denom, 0.0)
    df["chi2"]=np.round(chi2,4)
    df["pval"]=stats.chi2.sf(chi2,1)
    return df

tab=add_da_metrics(tab)
print("PRR/ROR/chi2 done. PRR range:",round(tab.prr.min(),2),"-",round(tab.prr.max(),2),
      "(no more 10^8 blowups)")

PRR/ROR/chi2 done. PRR range: 0.0 - 145401.39 (no more 10^8 blowups)


## 5. Empirical-Bayes Gamma-Poisson Shrinkage → real EBGM / EB05

The flagship metric. Replaces noisy small-count ratios with their Bayesian posterior.

### Algorithm

1. Fit a Gamma-Poisson prior to the entire collection of (drug, event) pairs.
2. For each pair, compute the posterior on the true rate ratio.
3. Output:
   - **EBGM** = posterior mean
   - **EB05** = 5th-percentile lower bound of the posterior (conservative signal strength)

Pairs with low counts (e.g. a=3) are shrunk strongly toward the prior, suppressing
false positives. Pairs with strong evidence move closer to their observed ratio. This is
DuMouchel's single-component approximation of the FDA's MGPS algorithm.

In [5]:
def fit_gamma_poisson(n,E):
    n=np.asarray(n,float); E=np.asarray(E,float)
    m=E>0; n,E=n[m],E[m]
    def nll(th):
        a,b=np.exp(th)
        ll=gammaln(n+a)-gammaln(a)-gammaln(n+1)+a*np.log(b/(b+E))+n*np.log(E/(b+E))
        return -np.sum(ll)
    rr=n/np.maximum(E,1e-9); mu=max(np.mean(rr),1e-2); var=np.var(rr)+1e-6
    a0=max(mu*mu/var,.1); b0=max(a0/mu,.1)
    try:
        res=minimize(nll,np.log([a0,b0]),method="Nelder-Mead",
                     options=dict(maxiter=3000,xatol=1e-4,fatol=1e-4))
        a,b=np.exp(res.x)
        if not (np.isfinite(a) and np.isfinite(b) and a>0 and b>0): raise ValueError
    except Exception:
        a,b=1.0,1.0
    return a,b

ALPHA,BETA=fit_gamma_poisson(tab.a.values,tab.expected.values)
print(f"Prior fit: alpha={ALPHA:.3f}, beta={BETA:.3f}, prior mean (RR)={ALPHA/BETA:.3f}")

post_shape=ALPHA+tab.a.values
post_rate =BETA+tab.expected.values
tab["ebgm"]=np.round(np.exp(digamma(post_shape)-np.log(post_rate)),4)
tab["eb05"]=np.round(stats.gamma.ppf(0.05,a=post_shape,scale=1.0/post_rate),4)
tab["eb95"]=np.round(stats.gamma.ppf(0.95,a=post_shape,scale=1.0/post_rate),4)
# Information component (BCPNN-style, shrunk)
ic=np.log2((tab.a.values+0.5)/(tab.expected.values+0.5))
tab["ic"]=np.round(ic,4)
tab["ic025"]=np.round(ic-1.96/np.sqrt(np.maximum(tab.a.values,1))/np.log(2),4)
print("EBGM range:",round(tab.ebgm.min(),2),"-",round(tab.ebgm.max(),2),
      "| EB05 range:",round(tab.eb05.min(),2),"-",round(tab.eb05.max(),2))

Prior fit: alpha=0.546, beta=0.090, prior mean (RR)=6.096
EBGM range: 0.0 - 589.25 | EB05 range: 0.0 - 510.7


## 6. FDR Correction + Signal Determination

Tens of thousands of 2x2 tables are tested simultaneously, so raw p-values are corrected
for multiple comparisons using Benjamini-Hochberg FDR (alpha=0.05).

A pair is flagged as a **signal** when:

- Primary criterion (MGPS standard): **EB05 > 4** AND FDR-significant.
- Supporting criteria: PRR ≥ 2, ROR CI lower bound > 1, IC025 > 0.

`signal_strength` is `Strong` if all four supporting criteria are met and EB05 >= 8,
otherwise `Moderate`.

In [ ]:
work=tab[tab.a>=MIN_COUNT].copy()
rej,padj,_,_=multipletests(work.pval.values,alpha=0.05,method="fdr_bh")
work["pval_adj"]=padj; work["fdr_sig"]=rej

work["crit_eb"]=(work.eb05>EB05_SIGNAL).astype(int)
work["crit_prr"]=((work.prr>=2)&work.fdr_sig).astype(int)
work["crit_ror"]=(work.ror_ci_lower>1).astype(int)
work["crit_ic"]=(work.ic025>0).astype(int)
work["criteria_met"]=work[["crit_eb","crit_prr","crit_ror","crit_ic"]].sum(axis=1)
# Primary criterion (MGPS): EB05 > threshold AND FDR-significant
work["is_signal"]=((work.eb05>EB05_SIGNAL)&work.fdr_sig)
work["signal_strength"]="None"
work.loc[work.is_signal,"signal_strength"]="Moderate"
work.loc[work.is_signal&(work.criteria_met==4)&(work.eb05>=8),"signal_strength"]="Strong"

tot=work.is_signal.sum()
print(f"Pairs tested (a>={MIN_COUNT}): {len(work):,}")
print(f"Signals: {tot:,} ({tot/len(work)*100:.1f}%) | "
      f"Strong={int((work.signal_strength=='Strong').sum()):,} "
      f"Moderate={int((work.signal_strength=='Moderate').sum()):,}")

## 7. Validation — Sensitivity on Modern Positive Controls

Sensitivity = fraction of established (drug, event) associations the pipeline correctly
flags as signals. A high sensitivity means the system catches what it should.

Note: pairs with `a < MIN_COUNT` in the modern data are not testable (insufficient
reports) and are excluded from the denominator. Withdrawn-era drugs (rofecoxib,
rosiglitazone) are validated separately in section 10 against the legacy population.

In [7]:
def lookup(df,drug,event):
    r=df[(df.target_drug==drug)&(df.pt==event)]
    if len(r)==0:
        r=df[(df.target_drug==drug)&(df.pt.str.contains(event,case=False,na=False))]
    return r.iloc[0] if len(r) else None

val_rows=[]
present_drugs=set(work.target_drug)
print("="*78); print("VALIDATION (modern positive controls)"); print("="*78)
for drug,events in KNOWN_SIGNALS.items():
    if drug not in present_drugs: continue          # legacy handled in §10
    for ev in events:
        r=lookup(work,drug,ev)
        if r is None:
            print(f"  ⚠️ {drug} → {ev}: pair not found (a<{MIN_COUNT}?)")
            val_rows.append(dict(drug=drug,event=ev,detected=False,eb05=np.nan,prr=np.nan,strength="missing"))
        else:
            mark="✅" if r.is_signal else "❌"
            print(f"  {mark} {drug} → {ev}: a={r.a:.0f} EBGM={r.ebgm:.1f} EB05={r.eb05:.1f} "
                  f"PRR={r.prr:.1f} [{r.signal_strength}]")
            val_rows.append(dict(drug=drug,event=ev,detected=bool(r.is_signal),
                                 eb05=float(r.eb05),prr=float(r.prr),strength=r.signal_strength))
val_df=pd.DataFrame(val_rows)
modern_val=val_df[val_df.strength!="missing"]
if len(modern_val):
    sens=modern_val.detected.mean()*100
    print(f"\nSensitivity (modern pos-controls): {modern_val.detected.sum()}/{len(modern_val)} = {sens:.0f}%")

VALIDATION (modern positive controls)
  ✅ ciprofloxacin → TENDON RUPTURE: a=60 EBGM=37.4 EB05=30.1 PRR=44.0 [Strong]
  ✅ ciprofloxacin → TENDON DISORDER: a=67 EBGM=70.3 EB05=57.3 PRR=95.6 [Strong]
  ✅ ciprofloxacin → TENDONITIS: a=125 EBGM=57.5 EB05=49.6 PRR=70.1 [Strong]
  ❌ ciprofloxacin → ARTHRALGIA: a=282 EBGM=3.3 EB05=3.0 PRR=3.3 [None]
  ✅ isotretinoin → DEPRESSION: a=254 EBGM=6.8 EB05=6.1 PRR=7.0 [Moderate]
  ✅ isotretinoin → CHEILITIS: a=49 EBGM=43.7 EB05=34.4 PRR=55.6 [Strong]
  ✅ isotretinoin → DRY SKIN: a=201 EBGM=5.3 EB05=4.7 PRR=5.4 [Moderate]
  ✅ clozapine → AGRANULOCYTOSIS: a=83 EBGM=7.1 EB05=5.9 PRR=7.6 [Moderate]
  ✅ clozapine → NEUTROPENIA: a=2768 EBGM=25.1 EB05=24.3 PRR=31.9 [Strong]
  ✅ clozapine → MYOCARDITIS: a=161 EBGM=16.8 EB05=14.7 PRR=19.8 [Strong]
  ✅ warfarin → HAEMORRHAGE: a=61 EBGM=9.6 EB05=7.8 PRR=9.9 [Moderate]
  ✅ warfarin → INTERNATIONAL NORMALISED RATIO INCREASED: a=128 EBGM=244.2 EB05=210.7 PRR=391.0 [Strong]
  ✅ warfarin → GASTROINTESTINAL HAEMORRHA

## 8. Specificity — Signal Rate by Group (headline check)

If the pipeline is well calibrated, the **negative-control group's signal rate should
be low.** A high negative-control signal rate would indicate the threshold is too
permissive (poor specificity).

In [ ]:
summ=(work.groupby(["target_drug","target_group","target_role"])
          .agg(pairs=("a","size"),signals=("is_signal","sum"),
               strong=("signal_strength",lambda s:(s=="Strong").sum()))
          .reset_index())
summ["signal_rate"]=np.round(summ.signals/summ.pairs*100,1)
summ=summ.sort_values(["target_group","signal_rate"],ascending=[True,False])
print(summ.to_string(index=False))

grp=(work.groupby("target_group")["is_signal"].agg(["size","sum"]))
grp["rate%"]=np.round(grp["sum"]/grp["size"]*100,1)
print("\n=== Signal rate by GROUP ===")
print(grp.to_string())
neg=grp.loc["negative_control","rate%"] if "negative_control" in grp.index else np.nan
print(f"\nNegative-control signal rate: {neg}%  (lower is better)")

## 8b. OMOP-style Negative Control — Pair-level Specificity

The group-level signal rate in section 8 counts **every** PT for a negative-control drug
as a candidate FP, inflating the rate because real adverse events appear there too. The
pharmacovigilance standard (OMOP / Ryan reference set) measures specificity at the
**verified-unrelated drug-event pair** level. The pairs below are well-established as
non-causal; if the pipeline flags one, it is a true false positive.

In [ ]:
# ============================================================================
# Section 8b. OMOP-style PAIR-LEVEL Negative Control Specificity
# ----------------------------------------------------------------------------
# The group-level negative-control rate from section 8 (e.g. ~28.7%) lumps all PTs
# for a negative-control drug together, so genuine adverse events for that drug
# inflate the apparent false-positive rate. The PV standard (OMOP / Ryan reference
# set) measures specificity on verified non-causal drug-event PAIRS instead.
# Pairs below are established as non-causal; a flagged signal here is a true FP.
# (Only MedDRA PTs that actually exist in this DB are listed.)
# ============================================================================
NEG_PAIRS = {
    "metformin":   ["TINNITUS","CATARACT","CONJUNCTIVITIS","ONYCHOMYCOSIS","NASAL CONGESTION","NEPHROLITHIASIS"],
    "lisinopril":  ["TINNITUS","CATARACT","NASAL CONGESTION","NEPHROLITHIASIS"],
    "omeprazole":  ["TINNITUS","CATARACT","CONJUNCTIVITIS","ONYCHOMYCOSIS","TONSILLITIS","NASAL CONGESTION","NEPHROLITHIASIS","PLANTAR FASCIITIS"],
    "amlodipine":  ["TINNITUS","CATARACT","NASAL CONGESTION","NEPHROLITHIASIS","PLANTAR FASCIITIS"],
    "amoxicillin": ["TINNITUS","CATARACT","CONJUNCTIVITIS","TONSILLITIS","NASAL CONGESTION","NEPHROLITHIASIS"],
}

omop_rows = []
print("=" * 80)
print("OMOP-STYLE NEGATIVE CONTROL  -  pair-level specificity")
print("Pairs below are non-causal by reference; is_signal=True is a true FP.")
print("=" * 80)
for drug, pts in NEG_PAIRS.items():
    if drug not in set(work.target_drug):
        continue
    for pt in pts:
        hit = work[(work.target_drug == drug) & (work.pt == pt)]
        if len(hit) == 0:
            omop_rows.append(dict(drug=drug, pt=pt, found=False, a=0,
                                  eb05=np.nan, prr=np.nan, is_signal=False, strength="absent"))
            continue
        r = hit.iloc[0]
        is_sig = bool(r.is_signal)
        omop_rows.append(dict(drug=drug, pt=pt, found=True, a=float(r.a),
                              eb05=float(r.eb05), prr=float(r.prr),
                              is_signal=is_sig, strength=r.signal_strength))
        mark = "FP!" if is_sig else " ok"
        print(f"  [{mark}] {drug:>12s} -> {pt:<22s}: "
              f"a={r.a:>4.0f} EB05={r.eb05:>7.2f} PRR={r.prr:>6.1f} [{r.signal_strength}]")

omop_df = pd.DataFrame(omop_rows)
ev = omop_df[omop_df.found]

print("\n" + "=" * 80)
print("SPECIFICITY  -  OMOP pair-level vs old group-level")
print("=" * 80)
old_grp = (work[work.target_group == "negative_control"]["is_signal"].mean()
           if "negative_control" in set(work.target_group) else float("nan"))
fp_eval = ev.is_signal.mean() if len(ev) else float("nan")
print(f"  OLD group-level FP rate (all neg-drug PTs) : {old_grp:.1%}")
print(f"  NEW OMOP pair-level FP rate (evaluable)    : {fp_eval:.1%}  "
      f"({int(ev.is_signal.sum())}/{len(ev)})")
print(f"  -> Specificity (OMOP)                      : {1-fp_eval:.1%}")

if ev.is_signal.any():
    print("\n  Residual FPs (likely age / comorbidity confounding, not drug-induced):")
    for _, r in ev[ev.is_signal].iterrows():
        print(f"    - {r.drug} -> {r.pt}: EB05={r.eb05:.1f} [{r.strength}]")
    print("  -> Disproportionality does not adjust for confounders; document as a limitation.")

omop_df.to_csv(os.path.join(RESULTS_DIR, "negative_control_omop.csv"), index=False)
print(f"\nSaved -> {os.path.join(RESULTS_DIR, 'negative_control_omop.csv')}")

## 9. Visualizations

In [ ]:
plotdf = work.copy()
plotdf["log2_ebgm"] = np.log2(plotdf.ebgm.clip(0.05, 2000))

# Convert FDR-adjusted p to -log10
plotdf["nlp"] = -np.log10(plotdf.pval_adj.clip(1e-300, 1))
plotdf["nlp"] = plotdf["nlp"].replace([np.inf, -np.inf], np.nan).fillna(300)

# Pairs whose p underflows to 0 all pile at nlp=300; spread them by EBGM rank
# between 300 and 500 so the volcano plot reveals their ordering.
ceiling_mask = plotdf["nlp"] >= 299
if ceiling_mask.sum() > 0:
    ebgm_rank = plotdf.loc[ceiling_mask, "ebgm"].rank(pct=True)
    plotdf.loc[ceiling_mask, "nlp"] = 300 + ebgm_rank * 200

y_max = plotdf["nlp"].max()

plt.figure(figsize=(9, 6))
for sig, col, lab in [(False,"#d5d8dc","Non-signal"),(True,"#e74c3c","Signal")]:
    s = plotdf[plotdf.is_signal == sig]
    plt.scatter(s.log2_ebgm, s.nlp, s=8, c=col, alpha=.5, label=lab)

plt.axhline(-np.log10(0.05), ls="--", c="red", lw=.8)
plt.axvline(1, ls="--", c="blue", lw=.8)

# Indicate where the underflow ceiling begins
plt.axhline(300, ls=":", c="grey", lw=.8, alpha=.5)
plt.text(plt.xlim()[1] * 0.95, 310, "p ~ 0 (spread by EBGM rank)",
         ha="right", fontsize=7, color="grey", style="italic")

plt.yscale("symlog", linthresh=50)
plt.ylim(0, y_max * 1.05)
plt.xlabel("log2(EBGM)")
plt.ylabel("-log10(FDR adj p)  [symlog scale]")
plt.legend(); plt.title("Volcano plot — EB-shrunk signals")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_volcano_v6.png"), dpi=120)
plt.show()

gp=grp.reset_index()
colors={"validation":"#e74c3c","gsk":"#2ecc71","trending":"#3498db","negative_control":"#95a5a6"}
plt.figure(figsize=(8,4))
plt.bar(gp.target_group,gp["rate%"],color=[colors.get(g,"#777") for g in gp.target_group])
plt.ylabel("signal rate (%)"); plt.title("Signal rate by group (negative_control should be low)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"08_signal_rate_group_v6.png"),dpi=120); plt.show()

if len(modern_val):
    piv=modern_val.assign(v=modern_val.detected.astype(int)).pivot_table(
        index="drug",columns="event",values="v",fill_value=-1)
    plt.figure(figsize=(10,3.5))
    plt.imshow(piv.values,cmap="RdYlGn",vmin=-1,vmax=1,aspect="auto")
    plt.xticks(range(len(piv.columns)),piv.columns,rotation=45,ha="right",fontsize=7)
    plt.yticks(range(len(piv.index)),piv.index,fontsize=8)
    plt.title("Validation heatmap (green = detected)"); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,"08_validation_heatmap_v6.png"),dpi=120); plt.show()

## 10. Legacy Validation (era-isolated, optional)

If Phase 1 produced `*_legacy` tables, validate rofecoxib and rosiglitazone using **only
the legacy population**. Modern and legacy data are never mixed.

In [ ]:
def has_table(conn,name):
    return pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name=?",conn,
                       params=[name]).shape[0]>0

conn=sqlite3.connect(DB_PATH)
legacy_ok=has_table(conn,"drug_legacy") and has_table(conn,"bg_event_marginals_legacy")
if legacy_ok:
    N_LEG=int(pd.read_sql("SELECT value FROM meta WHERE key='legacy_N'",conn).iloc[0,0])
    bgL=pd.read_sql("SELECT pt,n_reports FROM bg_event_marginals_legacy",conn); BGL=dict(zip(bgL.pt,bgL.n_reports))
    tgL=pd.read_sql("SELECT primaryid,target_drug,role_cod FROM drug_legacy WHERE is_target=1",conn)
    rcL=pd.read_sql("SELECT primaryid,pt FROM reac_legacy",conn)
    conn.close()
    tgL=tgL[tgL.role_cod.isin(DRUG_ROLES)].drop_duplicates(["primaryid","target_drug"])
    ndL=tgL.groupby("target_drug")["primaryid"].nunique()
    deL=rcL.merge(tgL,on="primaryid").drop_duplicates(["primaryid","target_drug","pt"])
    tL=deL.groupby(["target_drug","pt"])["primaryid"].nunique().reset_index(name="a")
    tL["n_drug"]=tL.target_drug.map(ndL); tL["n_event"]=tL.pt.map(BGL).fillna(tL.a).clip(lower=tL.a)
    tL["N"]=N_LEG; tL["b"]=tL.n_drug-tL.a; tL["c"]=tL.n_event-tL.a; tL["d"]=tL.N-tL.a-tL.b-tL.c
    tL["expected"]=(tL.n_drug*tL.n_event)/tL.N
    tL=tL[(tL.b>=0)&(tL.c>=0)&(tL.d>=0)&(tL.a>=MIN_COUNT)]
    tL=add_da_metrics(tL)
    aL,bL=fit_gamma_poisson(tL.a.values,tL.expected.values)
    ps=aL+tL.a.values; pr=bL+tL.expected.values
    tL["ebgm"]=np.exp(digamma(ps)-np.log(pr)); tL["eb05"]=stats.gamma.ppf(0.05,a=ps,scale=1/pr)
    tL["is_signal"]=tL.eb05>EB05_SIGNAL
    print("="*70); print(f"LEGACY VALIDATION (N={N_LEG:,})"); print("="*70)
    for drug in [d for d in KNOWN_SIGNALS if d in set(tL.target_drug)]:
        for ev in KNOWN_SIGNALS[drug]:
            r=tL[(tL.target_drug==drug)&(tL.pt.str.contains(ev,case=False,na=False))]
            if len(r):
                r=r.iloc[0]; mark="[OK]" if r.is_signal else "[--]"
                print(f"  {mark} {drug}->{ev}: a={r.a:.0f} EBGM={r.ebgm:.1f} EB05={r.eb05:.1f}")
            else:
                print(f"  [N/A] {drug}->{ev}: a<{MIN_COUNT} in legacy")
else:
    conn.close()
    print("Legacy tables not found — Phase 1 section 11 (RUN_LEGACY) was skipped. Skipping legacy validation.")

## 11. Save Results

In [12]:
keep=["target_drug","target_group","target_role","pt","a","b","c","d","expected","n_drug","n_event","N",
      "prr","ror","ror_ci_lower","ror_ci_upper","chi2","pval","pval_adj","fdr_sig",
      "ebgm","eb05","eb95","ic","ic025","criteria_met","is_signal","signal_strength"]
out=work[keep].rename(columns={"target_drug":"prod_ai","target_group":"group"})
conn=sqlite3.connect(DB_PATH)
out.to_sql("signal_results",conn,if_exists="replace",index=False)
conn.execute("INSERT OR REPLACE INTO meta VALUES (?,?)",("phase2_prior",json.dumps(dict(alpha=ALPHA,beta=BETA))))
conn.commit(); conn.close()

out.to_csv(os.path.join(RESULTS_DIR,"all_drug_event_metrics.csv"),index=False)
out[out.is_signal].to_csv(os.path.join(RESULTS_DIR,"detected_signals.csv"),index=False)
summ.to_csv(os.path.join(RESULTS_DIR,"drug_summary.csv"),index=False)
val_df.to_csv(os.path.join(RESULTS_DIR,"validation_report.csv"),index=False)
print("Saved: signal_results(DB) + 4 CSVs to", RESULTS_DIR)
print(f"  detected signals: {int(out.is_signal.sum()):,}")

Saved: signal_results(DB) + 4 CSVs to /content/drive/MyDrive/FAERS_Intelligence/results
  detected signals: 3,483


## 12. Phase 2 Complete

**Outputs:**
- `signal_results` table — includes EB-shrunk EBGM/EB05 from a single population (no
  v5-style blow-ups or constant collapse).
- Validation: modern positive-control sensitivity and negative-control specificity
  quantified.

**Next: Phase 3** — LLM severity classification cross-validated against the FAERS
outcome ground truth, with case summaries focused on top EB05 signals.

In [13]:
print("="*60); print("Phase 2 (v6) complete — EB-shrinkage signal detection"); print("="*60)
print(f"Pairs: {len(work):,} | Signals: {int(work.is_signal.sum()):,} "
      f"({work.is_signal.mean()*100:.1f}%)")
print(f"Prior: alpha={ALPHA:.2f} beta={BETA:.2f}")
print("Next: Phase 3 v6 (LLM severity x ground-truth death).")

Phase 2 (v6) complete — EB-shrinkage signal detection
Pairs: 14,750 | Signals: 3,483 (23.6%)
Prior: alpha=0.55 beta=0.09
Next: Phase 3 v6 (LLM severity x ground-truth death).
